In [ ]:
from datetime import datetime
import os
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
bucket = os.getenv('WORKSPACE_BUCKET')
bucket

In [ ]:
# upload GCTA (latest version (1.95.0) didn't work!)
!wget https://yanglab.westlake.edu.cn/software/gcta/bin/gcta-1.94.4-linux-kernel-3-x86_64.zip

In [ ]:
# copy chr22 plink files
!gsutil -u $$GOOGLE_PROJECT cp "gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/acaf_threshold_v7.1/plink_bed/*chr22*" .

In [ ]:
# copy ancestry prediction file
!gsutil -u $$GOOGLE_PROJECT cp "gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv" .

In [ ]:
# copy related samples
!gsutil -u $$GOOGLE_PROJECT cp "gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv" .

# Prepare discovery and target genotype

In [ ]:
anc = pd.read_csv('ancestry_preds.tsv', sep='\t')

In [ ]:
# summarize pop
!awk -F'\t' 'NR > 1 { print $2 }' ancestry_preds.tsv | sort | uniq -c

In [ ]:
# save pop info
!cut -f1,2 ancestry_preds.tsv | tail -n +2 > pop.tsv
!awk '$2 == "eur" {print $1}' pop.tsv > aou_EUR.txt

In [ ]:
# remove related samples
!awk 'FNR==NR {a[$1]; next} !($1 in a)' relatedness_flagged_samples.tsv aou_EUR.txt > aou_unrelated_EUR.txt

In [ ]:
# subset 50K EUR samples 
!shuf -n 50000 aou_unrelated_EUR.txt > aou_EUR_50K.txt

### QC and subset to 50K EUR only

In [ ]:
%%bash

# Identify ambiguous and duplicate SNPs 

awk '{if( ($5=="A" && $6=="T") || \
          ($5=="T" && $6=="A") || \
          ($5=="C" && $6=="G") || \
          ($5=="G" && $6=="C") ) print $2 }' acaf_threshold.chr22.bim > ambiguous_snps.txt

In [ ]:
%%bash

# Remove ambiguous and duplicate SNPs. only keep biallelic SNPs

plink2 \
  --bfile acaf_threshold.chr22 \
  --exclude ambiguous_snps.txt \
  --snps-only \
  --max-alleles 2 \
  --geno 0.01 \
  --rm-dup force-first \
  --keep aou_EUR_50K.txt \
  --make-bed \
  --out aou_chr22_no_amb_dup_qc

### Split dataset into 80%:20%

In [ ]:
%%bash

awk '{print $2}' aou_chr22_no_amb_dup_qc.fam > sample_list.txt
EIGHTY_PERCENT=$(($(wc -l < sample_list.txt) * 80 / 100))

# Shuffle and split into two files
shuf sample_list.txt > shuffled_temp.txt && 
head -n $EIGHTY_PERCENT shuffled_temp.txt > disc_samples.txt && 
tail -n +$((EIGHTY_PERCENT + 1)) shuffled_temp.txt > targ_samples.txt &&
rm shuffled_temp.txt

In [ ]:
%%bash

plink2 --bfile aou_chr22_no_amb_dup_qc \
    --keep disc_samples.txt \
    --make-bed \
    --out wgs_disc_chr22_qc 

plink2 --bfile aou_chr22_no_amb_dup_qc \
    --keep targ_samples.txt \
    --make-bed \
    --out wgs_targ_chr22_qc

In [ ]:
%%bash
  
# Discovery AF
plink2 \
    --bfile wgs_disc_chr22_qc  \
    --freq \
    --out wgs_disc_allele_freq_qc

# Target AF
plink2 \
    --bfile wgs_disc_chr22_qc  \
    --freq \
    --out wgs_targ_allele_freq_qc

In [ ]:
# AF distribution

disc_af = pd.read_csv('wgs_disc_allele_freq_qc.afreq', sep='\t')
sns.histplot(data=disc_af, x='ALT_FREQS', bins=30) 
plt.xlim(0, 0.5)

plt.title('Discovery AF Distribution')
plt.xlabel('AF')
plt.ylabel('Frequency')
plt.show()

In [ ]:
targ_af = pd.read_csv('wgs_targ_allele_freq_qc.afreq', sep='\t')
sns.histplot(data=targ_af, x='ALT_FREQS', bins=30) 
plt.xlim(0, 0.5)

plt.title('Target AF Distribution')
plt.xlabel('AF')
plt.ylabel('Frequency')
plt.show()

### MAF QC

In [ ]:
%%bash

plink2 --bfile wgs_disc_chr22_qc  \
    --maf 0.005 \
    --make-bed \
    --out wgs_disc_chr22_qc2 

plink2 --bfile wgs_targ_chr22_qc \
    --maf 0.005 \
    --make-bed \
    --out wgs_targ_chr22_qc2

In [ ]:
%%bash
  
# Discovery AF
plink2 \
    --bfile wgs_disc_chr22_qc2  \
    --freq \
    --out wgs_disc_allele_freq_qc2

# Target AF
plink2 \
    --bfile wgs_disc_chr22_qc2  \
    --freq \
    --out wgs_targ_allele_freq_qc2

## Target Array

In [ ]:
!gsutil -u $$GOOGLE_PROJECT cp "gs://fc-aou-datasets-controlled/v7/microarray/plink_v7.1/arrays.bim" .

In [ ]:
!awk '$1 == "chr22" {print $2}' arrays.bim > arrary_chr22.txt #25945

In [ ]:
%%bash

# extract GDA variants only 
plink2 --bfile wgs_targ_chr22_qc2 \
        --extract arrary_chr22.txt \
        --make-bed \
        --out array_targ_chr22_qc2

In [ ]:
%%bash

# get AF
plink2 --bfile array_targ_chr22_qc2 \
        --freq \
        --out array_targ_allele_freq_qc2

# Generate causal variant list

In [1]:
### Run below in R

In [ ]:
generate_causal_snps <- function(total_causal_snps, 
                                  array_bim_file = "array_targ_chr22_qc2.bim", 
                                  wgs_bim_file = "wgs_targ_chr22_qc2.bim", 
                                  allele_freq_file = "wgs_disc_allele_freq_qc.afreq",
                                  rsid_file = "rsid.txt",   # contains ID and rsID
                                  min_freq = 0.005, max_freq = 0.05) {

  library(dplyr)

  # sample sizes
  shared_sample_size_100 <- total_causal_snps  
  unique_sample_size_0   <- 0  
  shared_sample_size_75  <- floor(0.75 * total_causal_snps)
  unique_sample_size_25  <- total_causal_snps - shared_sample_size_75
  shared_sample_size_50  <- floor(0.50 * total_causal_snps)
  unique_sample_size_50  <- total_causal_snps - shared_sample_size_50
  shared_sample_size_25  <- floor(0.25 * total_causal_snps)
  unique_sample_size_75  <- total_causal_snps - shared_sample_size_25
  shared_sample_size_0   <- 0
  unique_sample_size_100 <- total_causal_snps

  # read data
  array_bim <- read.table(array_bim_file)
  wgs_bim   <- read.table(wgs_bim_file)
  allele_freq <- read.table(allele_freq_file, sep = "\t")
  rsid <- read.table(rsid_file, sep = "\t", header=TRUE) 

  # filter based on rsid IDs
  wgs_bim <- wgs_bim[wgs_bim$V2 %in% rsid$ID, ]
  allele_freq <- allele_freq[allele_freq$V2 %in% rsid$ID, ]

  # allele freq filter
  af_filtered <- allele_freq[allele_freq$V6 > min_freq & allele_freq$V6 < max_freq, ]

  # shared vs unique
  shared_snps <- wgs_bim[wgs_bim$V2 %in% array_bim$V2, ]
  unique_snps <- wgs_bim[!(wgs_bim$V2 %in% array_bim$V2), ]

  shared_filtered <- shared_snps[shared_snps$V2 %in% af_filtered$V2, ]
  unique_filtered <- unique_snps[unique_snps$V2 %in% af_filtered$V2, ]

  # merge with rsid to add rsID
  shared_filtered <- merge(shared_filtered, rsid, by.x="V2", by.y="ID")
  unique_filtered <- merge(unique_filtered, rsid, by.x="V2", by.y="ID")

  # sampling wrapper
  generate_samples <- function(shared_size, unique_size, prefix) {
    for (i in 1:10) {
      set.seed(122 + i)
      shared_sample <- shared_filtered[sample(nrow(shared_filtered), shared_size), ]
      unique_sample <- unique_filtered[sample(nrow(unique_filtered), unique_size), ]
      combined <- rbind(shared_sample, unique_sample)

      # create output folder if needed
      if (!dir.exists("causal_snplists")) dir.create("causal_snplists")

      # write ID list
      id_file <- file.path("causal_snplists",
                           paste0(prefix, "_", i, "_causal_ids.txt"))
      write.table(combined$V2, id_file,
                  row.names = FALSE, col.names = FALSE, quote = FALSE)

      # write rsID list
      rsid_file <- file.path("causal_snplists",
                             paste0(prefix, "_", i, "_causal_rsids.txt"))
      write.table(combined$rsID, rsid_file,
                  row.names = FALSE, col.names = FALSE, quote = FALSE)
    }
  }

  # generate all 5 sets
  generate_samples(shared_sample_size_100, unique_sample_size_0, "s1")
  generate_samples(shared_sample_size_75,  unique_sample_size_25, "s2")
  generate_samples(shared_sample_size_50,  unique_sample_size_50, "s3")
  generate_samples(shared_sample_size_25,  unique_sample_size_75, "s4")
  generate_samples(shared_sample_size_0,   unique_sample_size_100, "s5")
}

#
generate_causal_snps(total_causal_snps = 500)

In [ ]:
generate_causal_snps <- function(total_causal_snps,
                                  array_bim_file = "array_targ_chr22_qc2.bim", 
                                  wgs_bim_qc1_file = "wgs_targ_chr22_qc.bim", 
                                  wgs_bim_qc2_file = "wgs_targ_chr22_qc2.bim", 
                                  allele_freq_file = "wgs_disc_allele_freq_qc.afreq",
                                  rsid_file = "rsid.txt",   # contain ID + rsID
                                  min_freq_shared = 0.005, max_freq_shared = 0.05,
                                  max_freq_unique = 0.005) {


  # read files
  array_bim <- read.table(array_bim_file)
  wgs_qc1 <- read.table(wgs_bim_qc1_file)
  wgs_qc2 <- read.table(wgs_bim_qc2_file)
  allele_freq <- read.table(allele_freq_file, sep = "\t")
  rsid <- read.table(rsid_file, sep = "\t", header=TRUE) 

  # restrict to rsid IDs
  array_bim <- array_bim[array_bim$V2 %in% rsid$ID, ]
  wgs_qc1   <- wgs_qc1[wgs_qc1$V2 %in% rsid$ID, ]
  wgs_qc2   <- wgs_qc2[wgs_qc2$V2 %in% rsid$ID, ]
  allele_freq <- allele_freq[allele_freq$V2 %in% rsid$ID, ]

  # allele-frequency filtering
  af_shared <- allele_freq[allele_freq$V6 > min_freq_shared & allele_freq$V6 < max_freq_shared, ]
  af_unique <- allele_freq[allele_freq$V6 < max_freq_unique, ]

  # 1. Shared SNPs in both WGS QC2 and array 
  shared_snps <- wgs_qc2[wgs_qc2$V2 %in% array_bim$V2 & wgs_qc2$V2 %in% af_shared$V2, ]

  # 2. SNPs in WGS QC2 but not array (common-enough)
  unique_qc2_snps <- wgs_qc2[!(wgs_qc2$V2 %in% array_bim$V2) & wgs_qc2$V2 %in% af_shared$V2, ]

  # 3. Rare SNPs from QC1 only
  rare_snps <- wgs_qc1[
    !(wgs_qc1$V2 %in% array_bim$V2) &
    !(wgs_qc1$V2 %in% wgs_qc2$V2) &
      wgs_qc1$V2 %in% af_unique$V2,
  ]

  # add rsID by merging with rsid
  shared_snps <- merge(shared_snps, rsid, by.x = "V2", by.y = "ID")
  unique_qc2_snps <- merge(unique_qc2_snps, rsid, by.x = "V2", by.y = "ID")
  rare_snps <- merge(rare_snps, rsid, by.x = "V2", by.y = "ID")

  # ensure output dir exists
  if (!dir.exists("causal_snplists")) dir.create("causal_snplists")

  generate_samples <- function(p_shared, p_unique, p_rare, prefix) {
    for (i in 1:10) {
      set.seed(122 + i)

      n_shared <- floor(p_shared * total_causal_snps)
      n_unique <- floor(p_unique * total_causal_snps)
      n_rare   <- floor(p_rare   * total_causal_snps)

      shared_sample <- shared_snps[sample(nrow(shared_snps), n_shared), ]
      unique_sample <- unique_qc2_snps[sample(nrow(unique_qc2_snps), n_unique), ]
      rare_sample   <- rare_snps[sample(nrow(rare_snps), n_rare), ]

      combined <- rbind(shared_sample, unique_sample, rare_sample)

      # output: IDs
      id_file <- file.path("causal_snplists",
                           paste0(prefix, "_", i, "_causal_ids.txt"))
      write.table(combined$V2, id_file,
                  row.names = FALSE, col.names = FALSE, quote = FALSE)

      # output: rsIDs
      rsid_file <- file.path("causal_snplists",
                             paste0(prefix, "_", i, "_causal_rsids.txt"))
      write.table(combined$rsID, rsid_file,
                  row.names = FALSE, col.names = FALSE, quote = FALSE)
    }
  }

  # generate 5 scenarios
  generate_samples(0.50, 0,    0.50, prefix = "s6") #shared_sample_proportion, unique_qc2_sample_proportion, rare_sample_proportion
  generate_samples(0.25, 0,    0.75, prefix = "s7")
  generate_samples(0.25, 0.25, 0.50, prefix = "s8")
  generate_samples(0,    0.50, 0.50, prefix = "s9")
  generate_samples(0,    0.25, 0.75, prefix = "s10")
}

#
generate_causal_snps(total_causal_snps = 500)

# Simulate phenotype

## Discovery phenotype

In [ ]:
# Simulate quantitative traits with the heritability of 0.5

In [ ]:
%%bash

hsq=0.5
gcta_bin="gcta-1.94.4-linux-kernel-3-x86_64/gcta64"

for i in {1..10}; do
  for rep in {1..10}; do
    # Quantitative trait simulation
    $gcta_bin \
      --bfile wgs_disc_chr22_qc \
      --simu-qt \
      --simu-causal-loci causal_snplists/s${i}_${rep}_causal_ids.txt \
      --simu-hsq $hsq \
      --out disc_pheno/gcta_qt_s${i}_${rep}
  done
done

cd disc_pheno/

for i in {1..10}; do
  for rep in {1..10}; do
    echo "FID IID y" > gcta_qt_s${i}_${rep}.pheno
    cat gcta_qt_s${i}_${rep}.phen >> gcta_qt_s${i}_${rep}.pheno
  done
done

### Simulate PCs

In [ ]:
%%bash

plink2 --bfile wgs_disc_chr22_qc2 \
    --indep-pairwise 50 5 0.1 \
    --out wgs_disc_chr22_qc2
        
plink2 --bfile wgs_disc_chr22_qc2 \
   --extract wgs_disc_chr22_qc2.prune.in \
   --pca 10 approx \
   --out disc_pc/wgs_disc_chr22_qc2_LD_pruned
      
# write PC file
cd disc_pc
sed '1s/^#FID/FID/' wgs_disc_chr22_qc2_LD_pruned.eigenvec > PC.txt

## Target phenotype

In [ ]:
%%bash

hsq=0.5
gcta_bin="gcta-1.94.4-linux-kernel-3-x86_64/gcta64"

for i in {1..10}; do
  for rep in {1..10}; do
    # Extract causal effect sizes
    tail -n +2 disc_pheno/gcta_qt_s${i}_${rep}.par | awk '{print $1, $4}' > causal_beta/causal_snplist_s${i}_${rep}_qt.txt
    
    # Quantitative trait simulation
    $gcta_bin \
      --bfile wgs_targ_chr22_qc \
      --simu-qt \
      --simu-causal-loci causal_beta/causal_snplist_s${i}_${rep}_qt.txt \
      --simu-hsq $hsq \
      --out targ_pheno/gcta_qt_s${i}_${rep}
  done
done

cd targ_pheno/

for i in {1..10}; do
  for rep in {1..10}; do
    echo "FID IID y" > gcta_qt_s${i}_${rep}.pheno
    cat gcta_qt_s${i}_${rep}.phen >> gcta_qt_s${i}_${rep}.pheno
  done
done

### Simulate PCs

In [ ]:
%%bash

plink2 --bfile wgs_targ_chr22_qc2 \
    --indep-pairwise 50 5 0.1 \
    --out wgs_targ_chr22_qc2
        
plink2 --bfile wgs_targ_chr22_qc2 \
   --extract wgs_targ_chr22_qc2.prune.in \
   --pca 10 \
   --out targ_pc/wgs_targ_chr22_qc2_LD_pruned
      
# write PC file
cd targ_pc
sed '1s/^#FID/FID/' wgs_targ_chr22_qc2_LD_pruned.eigenvec > PC.txt

# Run GWAS

In [ ]:
%%bash

phe_dir="disc_pheno"
pc_dir="disc_pc"
out_dir="GWAS"

mkdir -p "$out_dir"  

for i in {1..10}; do
  reps=(1 2 3 4 5 6 7 8 9 10)

  # Run GWAS for quantitative traits
  for rep in "${reps[@]}"; do
    plink2 \
      --bfile "wgs_disc_chr22_qc2" \
      --pheno "${phe_dir}/gcta_qt_s${i}_${rep}.pheno" \
      --pheno-name y \
      --glm \
      --no-sex \
      --covar "${pc_dir}/PC.txt" \
      --out "${out_dir}/gcta_qt_s${i}.sumstats.rep${rep}"

    file="${out_dir}/gcta_qt_s${i}.sumstats.rep${rep}.y.glm.linear"
    
    if [[ -f "$file" ]]; then
      awk 'NR==1 || $10 == "ADD"' "$file" > "${file}.filtered"
      mv "${file}.filtered" "$file"
    else
      echo "Warning: Quantitative trait file not found: $file"
    fi
  done
done

# PRS

In [ ]:
%%bash

echo "1 0 1" >> range_list
echo "0.5 0 0.5" >> range_list
echo "0.1 0 0.1" >> range_list
echo "0.01 0 0.01" >> range_list
echo "0.001 0 0.001" >> range_list
echo "1e-04 0 1e-04" >> range_list
echo "1e-05 0 1e-05" >> range_list
echo "1e-06 0 1e-06" >> range_list
echo "1e-07 0 1e-07" >> range_list
echo "1e-08 0 1e-08" >> range_list

In [ ]:
%%bash

gwas_dir="GWAS"

declare -A geno_map=(
    [WGS]="wgs_targ_chr22_qc2"
    [Array]="array_targ_chr22_qc2"
)

for scenario in {1..10}; do
  for geno_type in WGS Array; do
    for rep in {1..10}; do  
      prs_dir="PRS/${geno_type}"
      mkdir -p "$prs_dir"

      input_file="${gwas_dir}/gcta_qt_s${scenario}.sumstats.rep${rep}.y.glm.linear"
      output_prefix="${prs_dir}/gcta_qt_s${scenario}.ct.rep${rep}"

      plink \
          --bfile "${geno_map[$geno_type]}" \
          --clump-p1 1 \
          --clump-r2 0.1 \
          --clump-kb 250 \
          --clump "$input_file" \
          --clump-snp-field ID \
          --clump-field P \
          --out "$output_prefix"

       awk 'NR!=1{print $3}' ${output_prefix}.clumped > ${output_prefix}.valid.snp
       awk '{print $3,$15}' "$input_file" > ${output_prefix}.SNP.pvalue 

       plink \
           --bfile "${geno_map[$geno_type]}" \
           --score "$input_file" 3 7 12 header \
           --q-score-range range_list ${output_prefix}.SNP.pvalue \
           --extract ${output_prefix}.valid.snp \
           --out ${output_prefix}.prs   
                
        done
    done
done

# R2 (run in R notebook)

In [ ]:
library(dplyr)

pcs_file <- file.path("targ_pc", "PC.txt")
pcs <- read.table(pcs_file, header = TRUE)

p.threshold <- c(1, 0.5, 0.1, 0.01, 0.001, 1e-04, 1e-05, 1e-06, 1e-07, 1e-08)

geno_types <- c("WGS", "Array")

prs_r2_results_list <- list()

for (i in 1:10) {  # scenario
  for (rep in 1:10) {  # replicate

    # --- Load phenotype and PCs ---
    pheno_file_qt <- file.path(
      "targ_pheno",
      paste0("gcta_qt_s", i, "_", rep, ".pheno")
    )
    phenotype_qt <- merge(
      read.table(pheno_file_qt, header = TRUE),
      pcs, by = c("FID", "IID")
    )

    for (geno in geno_types) {
      prs_dir <- file.path("PRS", geno)
      threshold_r2 <- data.frame()

      for (pthr in p.threshold) {
        prs_file <- file.path(
          prs_dir,
          paste0("gcta_qt_s", i, ".ct.rep", rep, ".prs.", pthr, ".profile")
        )

        if (!file.exists(prs_file)) next

        prs_data <- read.table(prs_file, header = TRUE)[, c("FID", "IID", "SCORE")]
        merged_data <- merge(phenotype_qt, prs_data, by = c("FID", "IID"))

        # --- Incremental R² (Full - Null) ---
        null_model <- lm(y ~ ., data = merged_data[, c("y", paste0("PC", 1:10))])
        full_model <- lm(y ~ ., data = merged_data[, c("y", paste0("PC", 1:10), "SCORE")])
        null_r2 <- summary(null_model)$r.squared
        full_r2 <- summary(full_model)$r.squared
        inc_r2 <- full_r2 - null_r2

        threshold_r2 <- rbind(
          threshold_r2,
          data.frame(Threshold = pthr, Null_R2 = null_r2, Full_R2 = full_r2, PRS_R2 = inc_r2)
        )
      }

      # --- Best threshold selection ---
      if (nrow(threshold_r2) > 0) {
        best_row <- threshold_r2[which.max(threshold_r2$PRS_R2), ]
        prs_r2_results_list[[length(prs_r2_results_list) + 1]] <- data.frame(
          Genotype = geno,
          Scenario = paste0("s", i),
          Replicate = rep,
          Best_Threshold = best_row$Threshold,
          PRS_R2 = best_row$PRS_R2,
          Null_R2 = best_row$Null_R2,
          Full_R2 = best_row$Full_R2
        )
      }
    }
  }
}

prs_r2_results <- bind_rows(prs_r2_results_list)
output_file <- file.path("PRS", "prs_r2_best_threshold.txt")

write.table(
  prs_r2_results,
  output_file,
  row.names = FALSE, col.names = TRUE, quote = FALSE, sep = "\t"
)